# 03 Búsqueda vectorial con TF-IDF sobre el corpus de Project Gutenberg (epub)

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

> **Nota de esta versión.** Esta práctica se resuelve con el corpus de libros del Proyecto Gutenberg descargado en `notebooks/epub/` (archivos `pg*.txt`), reutilizando el preprocesamiento y el muestreo reproducible de la práctica 02. Cada documento conserva su identificador original de Gutenberg.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg (carpeta `epub`) y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:
import math
import os
import random
import re
import time
import unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 170)

STOPWORDS = set('''a about above after again against ain all am an and any are aren aren't as at be because been before being below between both but by can couldn couldn't d did didn didn't do does doesn doesn't doing don don't down during each few for from further had hadn hadn't has hasn hasn't have haven haven't having he he'd he'll he's her here hers herself him himself his how i i'd i'll i'm i've if in into is isn isn't it it'd it'll it's its itself just ll m ma me mightn mightn't more most must mustn't my myself needn needn't no nor not now o of off on once only or other our ours ourselves out over own re s same shan shan't she she'd she'll she's should should've shouldn shouldn't so some such t than that that'll the their theirs them themselves then there these they they'd they'll they're they've this those through to too under until up ve very was wasn wasn't we we'd we'll we're we've were weren weren't what when where which while who whom why will with won won't wouldn wouldn't y you you'd you'll you're you've your yours yourself yourselves'''.split())
TOKEN_RE = re.compile(r'[a-z]+')


def normalize_text(text):
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch)).lower()
    start = text.find('*** start of')
    end = text.find('*** end of')
    if start != -1:
        text = text[start:]
    if end != -1:
        text = text[:end]
    return text


def tokenize(text):
    return [
        token
        for token in TOKEN_RE.findall(normalize_text(text))
        if len(token) >= 3 and token not in STOPWORDS
    ]


LARGE_ROOT = Path('epub')
if not LARGE_ROOT.exists():
    LARGE_ROOT = Path('notebooks') / 'epub'
if not LARGE_ROOT.exists():
    raise FileNotFoundError(
        'No se encontró la carpeta epub. Ejecute el notebook desde la raíz del proyecto o desde notebooks/.'
    )

epub_paths = sorted(LARGE_ROOT.glob('*/*.txt'), key=lambda path: int(path.stem[2:]))
print(f'Archivos TXT encontrados: {len(epub_paths):,}')


def extract_metadata(path):
    title = author = language = None
    with open(path, 'r', encoding='utf-8', errors='ignore') as handle:
        for _ in range(200):
            line = handle.readline()
            if not line:
                break
            stripped = line.strip()
            if stripped.startswith('Title:') and title is None:
                title = stripped[6:].strip()
            elif stripped.startswith('Author:') and author is None:
                author = stripped[7:].strip()
            elif stripped.startswith('Language:') and language is None:
                language = stripped[9:].strip()
            if title and author:
                break
    return title, author, language


max_workers = min(32, (os.cpu_count() or 1) + 4)
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    epub_metadata = list(executor.map(extract_metadata, epub_paths))


def normalized_key(text):
    text = (text or '').lower()
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return ' '.join(text.split())


def deduplication_key(path, title, author):
    key = (normalized_key(title), normalized_key(author))
    if not key[0] and not key[1]:
        key = ('__id__:' + path.parent.name, '')
    return key


records = []
seen_keys = {}
for path, (title, author, language) in zip(epub_paths, epub_metadata):
    doc_id = int(path.stem[2:])
    key = deduplication_key(path, title, author)
    is_duplicate = key in seen_keys
    if not is_duplicate:
        seen_keys[key] = path
    records.append(
        {
            'doc_id': doc_id,
            'path': path,
            'title': title,
            'author': author,
            'language': language,
            'bytes': path.stat().st_size,
            'duplicate': is_duplicate,
        }
    )

metadata_df = pd.DataFrame(records)
valid_df = metadata_df[~metadata_df['duplicate']].copy()
valid_df = valid_df.sort_values('doc_id').reset_index(drop=True)

SAMPLE_SIZE = 1000
rng = random.Random(42)
sample_positions = rng.sample(range(len(valid_df)), SAMPLE_SIZE)
working_df = valid_df.iloc[sample_positions].sort_values('doc_id').reset_index(drop=True)

doc_ids = [int(row.doc_id) for row in working_df.itertuples()]
paths = {int(row.doc_id): row.path for row in working_df.itertuples()}
titles = {
    int(row.doc_id): (
        row.title.strip()
        if isinstance(row.title, str) and row.title.strip()
        else f'pg{int(row.doc_id)}'
    )
    for row in working_df.itertuples()
}

start = time.perf_counter()
tokenized_docs = {}
for doc_id in doc_ids:
    text = paths[doc_id].read_text(encoding='utf-8', errors='ignore')
    tokenized_docs[doc_id] = tokenize(text)
load_seconds = time.perf_counter() - start

vocabulary = sorted({term for tokens in tokenized_docs.values() for term in tokens})

doc_table = pd.DataFrame(
    {
        'Identificador': doc_ids,
        'Título': [titles[doc_id] for doc_id in doc_ids],
        'Términos únicos': [len(set(tokenized_docs[doc_id])) for doc_id in doc_ids],
    }
)

print(f'Documentos TXT totales: {len(metadata_df):,}')
print(f'Documentos válidos tras deduplicar: {len(valid_df):,}')
print(f'Versiones duplicadas descartadas: {int(metadata_df["duplicate"].sum()):,}')
print(f'Muestra de trabajo: {len(doc_ids):,} documentos')
print(f'Bytes de la muestra: {working_df["bytes"].sum() / (1024 ** 2):,.2f} MB')
print(f'Tiempo de carga y tokenización: {load_seconds:,.2f} s')
print()
display(doc_table.head(30))
print(f'La tabla completa contiene {len(doc_table):,} filas (se muestran las primeras 30).')

doc_lengths = np.array([len(tokenized_docs[doc_id]) for doc_id in doc_ids])
print()
print(f'Número de documentos: {len(doc_ids):,}')
print(f'Tamaño del vocabulario: {len(vocabulary):,}')
print(f'Longitud mínima: {doc_lengths.min():,} términos')
print(f'Longitud media: {doc_lengths.mean():,.2f} términos')
print(f'Longitud máxima: {doc_lengths.max():,} términos')
print(f'Total de tokens procesados: {doc_lengths.sum():,}')

Archivos TXT encontrados: 78,145


Documentos TXT totales: 78,145
Documentos válidos tras deduplicar: 77,148
Versiones duplicadas descartadas: 997
Muestra de trabajo: 1,000 documentos
Bytes de la muestra: 340.28 MB
Tiempo de carga y tokenización: 54.74 s



,Identificador,Título,Términos únicos
0,59,Discourse on the Method of Rightly Conducting One's Reason and of Seeking Truth in the Sciences,2797
1,81,The Return of Tarzan,7028
2,163,Flower Fables,2343
3,257,Troilus and Criseyde,5205
4,318,John Barleycorn,7340
5,372,"Prince Otto, a Romance",7090
6,386,"Virginibus Puerisque, and Other Papers",7330
7,461,The Quest of the Golden Girl: A Romance,6205
8,501,The Story of Doctor Dolittle,2555
9,775,When the Sleeper Wakes,7819


La tabla completa contiene 1,000 filas (se muestran las primeras 30).

Número de documentos: 1,000
Tamaño del vocabulario: 857,038
Longitud mínima: 73 términos
Longitud media: 29,975.46 términos
Longitud máxima: 391,045 términos
Total de tokens procesados: 29,975,463


**Interpretación.** Se encontraron 78.145 archivos `pg*.txt`. Tras agrupar por título y autor para no contar distintas versiones del mismo libro, quedaron 77.148 documentos válidos (997 duplicados descartados). Para que el cálculo sea reproducible y tratable en memoria se tomó la misma muestra aleatoria de 1.000 documentos (`random_state=42`) que en la práctica 02. La muestra pesa 340,28 MB y reúne 29.975.463 tokens normalizados, con un vocabulario de 857.038 términos únicos.

Las longitudes son muy desiguales: el documento más corto tiene 73 términos y el más largo 391.045, con una media de 29.975,46. Esta diferencia anticipa que la normalización por longitud será importante al comparar documentos: un texto largo puede acumular muchas ocurrencias de un término sin ser necesariamente más relevante para la consulta. El corpus también es multilingüe (inglés, neerlandés, alemán, francés, etc.), lo que explica parte de la enorme variedad del vocabulario.

### 2. Implementación de funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html)

In [2]:
def compute_tf(tokens):
    return Counter(tokens)


def compute_df(tokenized_docs, vocabulary):
    df = {term: 0 for term in vocabulary}
    for tokens in tokenized_docs.values():
        for term in set(tokens):
            df[term] += 1
    return df


def compute_idf(df, n_documents, add_one=True):
    if add_one:
        return {
            term: math.log(n_documents / freq) + 1
            for term, freq in df.items()
        }
    return {
        term: math.log(n_documents / freq)
        for term, freq in df.items()
    }


def tf_weight(count, scheme, max_count):
    if count == 0:
        return 0.0
    if scheme == 'raw':
        return float(count)
    if scheme == 'log':
        return 1.0 + math.log(count)
    if scheme == 'augmented':
        return 0.5 + 0.5 * count / max_count
    raise ValueError(f'Esquema TF desconocido: {scheme}')


TF_SCHEMES = ('raw', 'log', 'augmented')
IDF_SCHEMES = ('classic', 'plus_one')
SIX_TFIDF_WEIGHTS = [
    (tf_scheme, idf_scheme)
    for tf_scheme in TF_SCHEMES
    for idf_scheme in IDF_SCHEMES
]

print('Seis pesos TF-IDF disponibles:')
for tf_scheme in TF_SCHEMES:
    for idf_scheme in IDF_SCHEMES:
        print(f'  TF={tf_scheme:9s} IDF={idf_scheme}')

sample_doc = doc_ids[0]
sample_counts = compute_tf(tokenized_docs[sample_doc])
sample_term = sample_counts.most_common(1)[0][0]
sample_max_count = max(sample_counts.values())
df_all = compute_df(tokenized_docs, vocabulary)
idf_classic = compute_idf(df_all, len(doc_ids), add_one=False)
idf_plus_one = compute_idf(df_all, len(doc_ids), add_one=True)

rows = []
for tf_scheme in TF_SCHEMES:
    for idf_scheme in IDF_SCHEMES:
        idf_values = idf_plus_one if idf_scheme == 'plus_one' else idf_classic
        rows.append(
            {
                'Documento': sample_doc,
                'Término': sample_term,
                'TF': tf_scheme,
                'IDF': idf_scheme,
                'Peso TF-IDF': tf_weight(sample_counts[sample_term], tf_scheme, sample_max_count)
                * idf_values[sample_term],
            }
        )
display(pd.DataFrame(rows))

Seis pesos TF-IDF disponibles:
  TF=raw       IDF=classic
  TF=raw       IDF=plus_one
  TF=log       IDF=classic
  TF=log       IDF=plus_one
  TF=augmented IDF=classic
  TF=augmented IDF=plus_one


,Documento,Término,TF,IDF,Peso TF-IDF
0,59,one,raw,classic,0.000000
1,59,one,raw,plus_one,80.000000
2,59,one,log,classic,0.000000
3,59,one,log,plus_one,5.382027
4,59,one,augmented,classic,0.000000
5,59,one,augmented,plus_one,1.000000


**Interpretación.** Los seis pesos combinan tres formas de TF (`raw`, `log`, `augmented`) con dos formas de IDF (clásica y `+1`). El TF crudo utiliza el conteo directo; el logarítmico comprime las repeticiones; el aumentado normaliza por la frecuencia máxima del documento. La IDF clásica es `ln(N/DF)` y la IDF `+1` es `ln(N/DF)+1`. Para la matriz del corpus usamos TF crudo e IDF `+1`, que es la configuración que reproduce `TfidfVectorizer` con `smooth_idf=False`.

El término de ejemplo (`one`) es el más frecuente del primer documento de la muestra y aparece en los 1.000 documentos. Por eso su IDF clásica es `ln(1000/1000)=0` y todos los pesos con IDF clásica valen cero; con IDF `+1` el peso pasa a ser igual al TF. Este caso extremo muestra por qué la variante `+1` evita que un término presente en todos los documentos tenga peso nulo, y por qué los términos poco frecuentes son los que realmente discriminan.

### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

La matriz se construye en formato **disperso** (`scipy.sparse.csr_matrix`) porque el vocabulario tiene cientos de miles de términos y solo una pequeña fracción aparece en cada documento.

In [3]:
vocabulary = sorted({term for tokens in tokenized_docs.values() for term in tokens})
term_index = {term: i for i, term in enumerate(vocabulary)}

tf_counts = {doc_id: compute_tf(tokenized_docs[doc_id]) for doc_id in doc_ids}

df = compute_df(tokenized_docs, vocabulary)

idf = compute_idf(df, len(doc_ids), add_one=True)

rows, cols, values = [], [], []
for row, doc_id in enumerate(doc_ids):
    for term, count in tf_counts[doc_id].items():
        rows.append(row)
        cols.append(term_index[term])
        values.append(count * idf[term])

tfidf_matrix = sparse.csr_matrix(
    (values, (rows, cols)),
    shape=(len(doc_ids), len(vocabulary)),
)

print(f'Documentos (filas): {tfidf_matrix.shape[0]:,}')
print(f'Términos (columnas): {tfidf_matrix.shape[1]:,}')
print(f'Valores no nulos: {tfidf_matrix.nnz:,}')
print(
    'Densidad: '
    f'{tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4%}'
)
print(f'Máximo peso TF-IDF: {tfidf_matrix.max():,.4f}')

coo = tfidf_matrix.tocoo()
max_position = int(coo.data.argmax())
max_row, max_col = int(coo.row[max_position]), int(coo.col[max_position])
max_term = vocabulary[max_col]
max_doc = doc_ids[max_row]
print(
    f'Peso máximo: término {max_term!r} en {titles[max_doc]!r} '
    f'(TF={tf_counts[max_doc][max_term]:,}, DF={df[max_term]:,}, '
    f'IDF={idf[max_term]:.4f})'
)

sample_terms = ['love', 'whale', 'king', 'poems']
sample_rows = []
for term in sample_terms:
    column = tfidf_matrix[:, term_index[term]].toarray().ravel()
    best = int(column.argmax())
    sample_rows.append(
        {
            'Término': term,
            'DF': df[term],
            'IDF': round(idf[term], 4),
            'Peso máximo': round(float(column[best]), 4),
            'Documento con mayor peso': titles[doc_ids[best]],
        }
    )
display(pd.DataFrame(sample_rows))

Documentos (filas): 1,000
Términos (columnas): 857,038
Valores no nulos: 6,069,674
Densidad: 0.7082%
Máximo peso TF-IDF: 44,698.0931
Peso máximo: término 'subst' en 'Engelsch woordenboek. Eerste deel: Engelsch-Nederlandsch' (TF=6,854, DF=4, IDF=6.5215)


,Término,DF,IDF,Peso máximo,Documento con mayor peso
0,love,677,1.3901,532.4022,The Duke's Children
1,whale,107,3.2349,1462.1868,"Whales, dolphins, and porpoises of the western North Atlantic"
2,king,588,1.5310,1382.5186,Stories of Charlemagne and the twelve Peers of France
3,poems,182,2.7037,156.8174,The Works of Lord Byron: Letters and Journals. Vol. 2


**Interpretación.** La matriz tiene 1.000 filas y 857.038 columnas, con 6.069.674 celdas no nulas: apenas el 0,708% del total, por lo que la representación es muy dispersa (99,29% de ceros). Por eso se usa una matriz CSR: almacenar la matriz densa requeriría más de 6 GB.

El peso máximo (44.698,09) corresponde a `subst` en *Engelsch woordenboek. Eerste deel: Engelsch-Nederlandsch*: aparece 6.854 veces en ese documento y solo en 4 de los 1.000 documentos (IDF ≈ 6,52). Esto muestra que el peso combina la frecuencia local con la rareza global. Más adelante, la similitud coseno normalizará la longitud de cada documento para que los textos más extensos no dominen únicamente por acumular ocurrencias.

### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tokenized_corpus = [tokenized_docs[doc_id] for doc_id in doc_ids]

vectorizer = TfidfVectorizer(
    analyzer=lambda tokens: tokens,
    lowercase=False,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
)
tfidf_sklearn = vectorizer.fit_transform(tokenized_corpus)

sklearn_terms = list(vectorizer.get_feature_names_out())
sklearn_index = {term: i for i, term in enumerate(sklearn_terms)}
manual_matrix_reordered = tfidf_matrix[:, [term_index[term] for term in sklearn_terms]]

difference = manual_matrix_reordered - tfidf_sklearn
max_difference = 0.0 if difference.nnz == 0 else float(np.abs(difference.data).max())

print(f'Términos en el vectorizador: {len(sklearn_terms):,}')
print(f'Forma de la matriz manual: {tfidf_matrix.shape}')
print(f'Forma de la matriz de scikit-learn: {tfidf_sklearn.shape}')
print(f'Máxima diferencia absoluta: {max_difference:.3e}')

assert max_difference <= 1e-10
print('Las dos representaciones coinciden dentro de la tolerancia (atol=1e-10).')

example_doc = doc_ids[int(np.argmax(tfidf_matrix[:, term_index['whale']].toarray().ravel()))]
example_row = doc_ids.index(example_doc)
comparison_rows = []
for term in ['love', 'whale', 'king', 'poems']:
    comparison_rows.append(
        {
            'Término': term,
            'IDF manual': idf[term],
            'IDF scikit-learn': vectorizer.idf_[sklearn_index[term]],
            'Peso manual': float(tfidf_matrix[example_row, term_index[term]]),
            'Peso scikit-learn': float(tfidf_sklearn[example_row, sklearn_index[term]]),
        }
    )

print(f'Documento de ejemplo: {titles[example_doc]!r}')
display(pd.DataFrame(comparison_rows))

Términos en el vectorizador: 857,038
Forma de la matriz manual: (1000, 857038)
Forma de la matriz de scikit-learn: (1000, 857038)
Máxima diferencia absoluta: 0.000e+00
Las dos representaciones coinciden dentro de la tolerancia (atol=1e-10).
Documento de ejemplo: 'Whales, dolphins, and porpoises of the western North Atlantic'


,Término,IDF manual,IDF scikit-learn,Peso manual,Peso scikit-learn
0,love,1.390084,1.390084,0.000000,0.000000
1,whale,3.234926,3.234926,1462.186753,1462.186753
2,king,1.531028,1.531028,0.000000,0.000000
3,poems,2.703749,2.703749,0.000000,0.000000


**Interpretación.** Las dos matrices tienen la misma forma y la máxima diferencia absoluta es 0,0. La coincidencia es exacta para esta configuración: se usaron los mismos tokens, las columnas se alinearon con el orden del vectorizador y la IDF calculada es `ln(N/DF)+1`. También se confirma que `norm=None` y `sublinear_tf=False` están desactivados. Cualquier diferencia de punto flotante queda muy por debajo de la tolerancia de `1e-10`.

### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

**Consultas definidas y necesidad de información**

| Consulta | Texto de la consulta | Qué espera encontrar el usuario |
|---|---|---|
| Amor y matrimonio | `love marriage family` | Novelas de costumbres centradas en relaciones amorosas, matrimonio y familia. |
| Ballena y mar | `whale sea captain` | Relatos marítimos y de caza de ballenas; textos sobre navegación y capitanes. |
| Poder y corona | `king queen crown` | Obras sobre realeza, poder político y sucesión; historia, memorias y tragedias. |
| Poesía y muerte | `poems soul death` | Poesía y reflexiones sobre el alma, la muerte y la trascendencia. |

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

queries = {
    'amor y matrimonio': 'love marriage family',
    'ballena y mar': 'whale sea captain',
    'poder y corona': 'king queen crown',
    'poesía y muerte': 'poems soul death',
}

doc_norms = np.sqrt(tfidf_matrix.multiply(tfidf_matrix).sum(axis=1)).A1


def build_query_vector(query):
    tokens = tokenize(query)
    counts = Counter(tokens)
    vector = np.zeros(len(vocabulary), dtype=float)
    for term, count in counts.items():
        index = term_index.get(term)
        if index is not None:
            vector[index] = count * idf[term]
    return tokens, vector


def manual_cosine_similarity(query_vector, doc_matrix):
    dot_products = np.asarray(doc_matrix @ query_vector).ravel()
    query_norm = float(np.linalg.norm(query_vector))
    denominator = query_norm * doc_norms
    similarities = np.divide(
        dot_products,
        denominator,
        out=np.zeros_like(dot_products),
        where=denominator != 0,
    )
    return dot_products, query_norm, doc_norms, similarities


query_data = {}
summary_rows = []
for label, query in queries.items():
    tokens, vector = build_query_vector(query)
    dot_products, query_norm, _, manual_scores = manual_cosine_similarity(
        vector, tfidf_matrix
    )
    library_scores = cosine_similarity(vectorizer.transform([tokens]), tfidf_sklearn)[0]
    difference = float(np.max(np.abs(manual_scores - library_scores)))
    best_position = int(np.argmax(manual_scores))
    query_data[label] = {
        'query': query,
        'tokens': tokens,
        'vector': vector,
        'dot_products': dot_products,
        'query_norm': query_norm,
        'doc_norms': doc_norms,
        'manual_scores': manual_scores,
        'library_scores': library_scores,
    }
    summary_rows.append(
        {
            'Consulta': label,
            'Texto': query,
            'Tokens procesados': ' '.join(tokens),
            'En vocabulario': f'{sum(term in term_index for term in tokens)}/{len(tokens)}',
            'Mejor documento': titles[doc_ids[best_position]],
            'Mejor similitud': round(float(manual_scores[best_position]), 6),
            'Máx. diferencia con sklearn': f'{difference:.3e}',
        }
    )

display(pd.DataFrame(summary_rows))

example_label = 'ballena y mar'
example = query_data[example_label]
best_position = int(np.argmax(example['manual_scores']))
best_doc = doc_ids[best_position]

shared_terms = sorted(
    term
    for term in set(example['tokens'])
    if term in term_index and tfidf_matrix[best_position, term_index[term]] != 0
)
partial_rows = []
partial_sum = 0.0
for term in shared_terms:
    index = term_index[term]
    query_weight = example['vector'][index]
    doc_weight = float(tfidf_matrix[best_position, index])
    product = query_weight * doc_weight
    partial_sum += product
    partial_rows.append(
        {
            'Término compartido': term,
            'TF-IDF consulta': query_weight,
            'TF-IDF documento': doc_weight,
            'Producto parcial': product,
        }
    )

partial_table = pd.DataFrame(partial_rows).sort_values('Producto parcial', ascending=False)
print(f'Consulta: {example["query"]!r}')
print(f'Documento: {best_doc} ({titles[best_doc]})')
display(partial_table)
print(f'Producto punto (suma de productos parciales): {partial_sum:.6f}')
print(f'Producto punto (q · d): {example["dot_products"][best_position]:.6f}')
print(f'Norma de la consulta: {example["query_norm"]:.6f}')
print(f'Norma del documento: {example["doc_norms"][best_position]:.6f}')
print(f'Similitud manual: {example["manual_scores"][best_position]:.6f}')
print(f'Similitud scikit-learn: {example["library_scores"][best_position]:.6f}')

,Consulta,Texto,Tokens procesados,En vocabulario,Mejor documento,Mejor similitud,Máx. diferencia con sklearn
0,amor y matrimonio,love marriage family,love marriage family,3/3,Friendship,0.145861,1.152e-15
1,ballena y mar,whale sea captain,whale sea captain,3/3,Four years aboard the whaleship,0.274002,1.832e-15
2,poder y corona,king queen crown,king queen crown,3/3,"Kensington Palace, the birthplace of the Queen",0.271156,2.991e-15
3,poesía y muerte,poems soul death,poems soul death,3/3,"The medical aspects of death, and the medical aspects of the human mind",0.127796,2.359e-15


Consulta: 'whale sea captain'
Documento: 68958 (Four years aboard the whaleship)


,Término compartido,TF-IDF consulta,TF-IDF documento,Producto parcial
2,whale,3.234926,679.334553,2197.597311
0,captain,1.860383,362.774704,674.899929
1,sea,1.430783,198.878825,284.552426


Producto punto (suma de productos parciales): 3157.049666
Producto punto (q · d): 3157.049666
Norma de la consulta: 3.996613
Norma del documento: 2882.938815
Similitud manual: 0.274002
Similitud scikit-learn: 0.274002


**Lectura del ejemplo.** Para la consulta `whale sea captain` y *Four years aboard the whaleship*, los tres términos son compartidos. El término `whale` aporta la mayor parte del producto punto (2.197,60 de 3.157,05), seguido de `captain` (674,90) y `sea` (284,55). El producto punto coincide con la suma de los productos parciales; al dividirlo entre las normas (3,9966 para la consulta y 2.882,94 para el documento) se obtiene una similitud de 0,274002. La diferencia con `cosine_similarity` es del orden de `1e-15`, atribuible al redondeo de punto flotante. El vector de la consulta se construyó con el vocabulario y los IDF del corpus; la consulta no se añadió como documento, por lo que su IDF no se recalculó.

### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [6]:
total_term_frequency = Counter()
for counts in tf_counts.values():
    total_term_frequency.update(counts)

raw_cache = {}


def read_raw(doc_id):
    if doc_id not in raw_cache:
        text = paths[doc_id].read_text(encoding='utf-8', errors='ignore')
        start = re.search(r'\*\*\*\s*START OF', text, flags=re.IGNORECASE)
        if start:
            newline = text.find('\n', start.end())
            text = text[newline + 1:] if newline != -1 else text[start.end():]
        end = re.search(r'\*\*\*\s*END OF', text, flags=re.IGNORECASE)
        if end:
            text = text[:end.start()]
        raw_cache[doc_id] = text
    return raw_cache[doc_id]


def extract_snippet(raw_text, query_tokens, width=220):
    ordered_terms = sorted(
        query_tokens,
        key=lambda term: (-idf.get(term, 0.0), total_term_frequency.get(term, 0)),
    )
    for term in ordered_terms:
        match = re.search(rf'\b{re.escape(term)}\b', raw_text, flags=re.IGNORECASE)
        if match:
            start = max(0, match.start() - width // 3)
            end = min(len(raw_text), start + width)
            if start > 0:
                next_space = raw_text.find(' ', start)
                if next_space != -1 and next_space < match.start():
                    start = next_space + 1
            if end < len(raw_text):
                previous_space = raw_text.rfind(' ', start, end)
                if previous_space != -1 and previous_space > match.end():
                    end = previous_space
            snippet = raw_text[start:end]
            break
    else:
        snippet = raw_text[:width]
    return re.sub(r'\s+', ' ', snippet).strip()


pd.set_option('display.max_colwidth', 220)
tolerance = 1e-10
ranking_differences = []

for label, query in queries.items():
    data = query_data[label]
    order = np.argsort(-data['manual_scores'])[:5]
    ranking_rows = []
    for position, doc_position in enumerate(order, start=1):
        doc_id = doc_ids[doc_position]
        ranking_rows.append(
            {
                'Posición': position,
                'Identificador': doc_id,
                'Título': titles[doc_id],
                'Puntuación': round(float(data['manual_scores'][doc_position]), 6),
                'Fragmento': extract_snippet(read_raw(doc_id), data['tokens']),
            }
        )
    maximum_difference = float(np.max(np.abs(data['manual_scores'] - data['library_scores'])))
    ranking_differences.append(maximum_difference)
    print(f'Consulta: {query!r} — {label}')
    display(pd.DataFrame(ranking_rows))

print(f'Máxima diferencia global entre implementación explícita y scikit-learn: {max(ranking_differences):.3e}')
assert max(ranking_differences) <= tolerance
print(f'Todas las puntuaciones coinciden dentro de la tolerancia (atol={tolerance:g}).')

Consulta: 'love marriage family' — amor y matrimonio


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,20861,Friendship,0.145861,is less thought of among us now than it was in the old world. Marriage has come to mean infinitely more. Communion with God in Christ has become to multitudes the primal fact of life. Nevertheless the need
1,2,44891,"The Works of the Right Honourable John, Earl of Rochester",0.144940,"dixi pede, &c. imitated._ p. 19 _A Satire against Marriage._ p. 25 _A Letter from _Artemisa_ in the Town, to _Cloe_ in the Country._"
2,3,33943,Woman,0.144098,"the horizon, it is that she is thinking of me: ""But that is what marriage means, my little daughter; you'll see, it is just a big renunciation: you will change, you too, and do like the rest; look at me; am I"
3,4,36915,His Lady of the Sonnets,0.118264,"lives, God willing: All the gifts from all the heights, your own, Love!"" ROBERT BROWNING. CONTENTS His Lady Of The Sonnets Antony To Cleopatra, After Actium"
4,5,27349,Personal Friendships of Jesus,0.117589,"as well as for sunny paths and days of gladness and song. He went to a marriage feast, and wrought his first miracle to prolong the festivity; but he went also to the home of grief, and turned its sorrow into joy. It"


Consulta: 'whale sea captain' — ballena y mar


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,68958,Four years aboard the whaleship,0.274002,"ceteras belonging to the whaleman’s pursuit; in fact, the yield of the whale supports New Bedford, and is the nucleus around which clusters all the manufactures of the city; and its vitality as a community must ever"
1,2,33527,"Whales, dolphins, and porpoises of the western North Atlantic",0.232971,"he encountered in that area and was intended for use in two ongoing whale observer programs, NUC's Whale Watch and NMFS's Platforms of Opportunity. The rationale of these programs was that since oceanographers,"
2,3,34634,A Narrative of the Death of Captain James Cook,0.155142,of the Death OF Captain James Cook By DAVID SAMWELL Surgeon of The Discovery LONDON:
3,4,57882,"A Cruise in the Sky; or, The Legend of the Great Pink Pearl",0.117332,TAKES A DARING CHANCE 157 XV TIMBADO KEY AND CAPTAIN MONCKTON BASSETT 171 XVI THE CANNIBAL KING AND THE PINK PEARL 186 XVII THE BIRD OF DEATH
4,5,67899,"The Motor Rangers on Blue Water; or, The Secret of the Derelict",0.107368,"of the sea and indeed rather mistrusted it, looked downcast. ""Maybe a whale swallered yer ship, Nat,"" he ventured. In spite of his chagrin over the non-arrival of the ""Nomad,"" Nat had to laugh, for Cal had made the"


Consulta: 'king queen crown' — poder y corona


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,43428,"Kensington Palace, the birthplace of the Queen",0.271156,transcriber.)] =Kensington Palace: the Birthplace of the Queen.= =Illustrated.= [Illustration: THIS MINIATURE REPRESENTS THE QUEEN AT THE AGE OF EIGHT]
1,2,46330,The Introduction to Hegel's Philosophy of Fine Arts,0.213439,"and Jewel.= Containing 'In an Indian Temple,' 'A Casket of Gems,' 'A Queen's Revenge,' with other Poems. Third Edition. Crown 8vo, 6_s._ =Pearls of the Faith=; or, Islam's Rosary. Being the Ninetynine Beautiful"
2,3,3884,"Memoirs of the Court of Marie Antoinette, Queen of France, Volume 1",0.185536,"by David Widger MEMOIRS OF THE COURT OF MARIE ANTOINETTE, QUEEN OF FRANCE Being the Historic Memoirs of Madam Campan, First Lady in Waiting to the Queen LIST OF ILLUSTRATIONS Duchesse du"
3,4,75339,Stories of Charlemagne and the twelve Peers of France,0.180669,"hold of him, having one of the chess pieces in his hand, to wit, the Queen, with which he was about to give mate to Yonnet, he drew back his arm and gave one of the men such a buffet on the head as killed him"
4,5,18299,"The Norwegian account of Haco's expedition against Scotland, A.D. MCCLXIII.",0.174830,"the neighbouring firth had its name.] [Footnote 58: _i.e._ John the Queen, perhaps the ancestor of the McQueens.] [Footnote 59: Asleifarvik (_orig._). Fl. MS., Hals-eyiar-vic.] [Footnote 60: _i.e._ The old"


Consulta: 'poems soul death' — poesía y muerte


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,79198,"The medical aspects of death, and the medical aspects of the human mind",0.127796,"“If I were now to die, ’Twere now to be most happy; for I fear My soul hath her content so absolute, That not another comfort like to this Succeeds in unknown fate.” I might bring forward many other"
1,2,8161,Fragments of Ancient Poetry,0.122997,"that followed, the body of poetry that was eventually collected as _The Poems of Ossian_ provoked the comment of nearly every important man of letters. Extravagance and partisanship were characteristic of most of"
2,3,63399,"The Glebe 1914/09 (Vol. 2, No. 2): Poems",0.087024,THIS ISSUE 50 CENTS POEMS George Cronyn The only editorial policy of THE GLEBE is that embodied in its declaration of absolute freedom
3,4,52865,Home Poems,0.083696,"HOME POEMS, — BY — KATE LOUISE WHEELER. Copyright 1897, BY KATE LOUISE"
4,5,23204,"Notes and Queries, Number 70, March 1, 1851",0.083056,"edition of his poetical or dramatic works. The current editions of his poems may therefore be open to censure, both on the score of deficiency and redundancy--and such I believe to be the fact. An _Essay on satire_,"


Máxima diferencia global entre implementación explícita y scikit-learn: 2.991e-15
Todas las puntuaciones coinciden dentro de la tolerancia (atol=1e-10).


**Análisis del ranking**

- **`love marriage family`** — El usuario buscaba novelas de costumbres y relaciones amorosas. El ranking lo encabezan *Friendship* (0,1459), *The Works of the Right Honourable John, Earl of Rochester* (0,1449) y *Woman* (0,1441), seguidos de *His Lady of the Sonnets* (0,1183) y *Personal Friendships of Jesus* (0,1176). Las puntuaciones son muy parecidas: los tres términos son frecuentes en el corpus (DF de 677, 433 y 638 respectivamente), así que la IDF es baja y la similitud depende sobre todo de la densidad relativa de esos términos en cada documento. El sistema recupera obras sobre amistad, matrimonio y relaciones, pero no distingue el género novela ni aplica stemming.
- **`whale sea captain`** — *Four years aboard the whaleship* queda primero con 0,2740, seguido de *Whales, dolphins, and porpoises of the western North Atlantic* (0,2330) y *A Narrative of the Death of Captain James Cook* (0,1551). El primero concentra `whale`, `sea` y `captain` en un relato de caza de ballenas; el segundo es un texto científico sobre cetáceos; el tercero comparte `sea` y `captain` por su contexto naval. Los dos últimos puestos son relatos de aventuras marítimas que comparten vocabulario pero no el tema central. El ranking es temáticamente coherente, aunque los puestos 4 y 5 son falsos positivos por solapamiento léxico.
- **`king queen crown`** — *Kensington Palace, the birthplace of the Queen* (0,2712), *The Introduction to Hegel's Philosophy of Fine Arts* (0,2134) y *Memoirs of the Court of Marie Antoinette, Queen of France, Volume 1* (0,1855). El primero y el tercero son textos históricos sobre la realeza; el segundo entra por el uso de `king`, `queen` y `crown` en un contexto filosófico y artístico. *Stories of Charlemagne and the twelve Peers of France* (0,1807) y la crónica de la expedición de Haco contra Escocia (0,1748) completan un ranking dominado por historia y memorias, no por tragedia.
- **`poems soul death`** — *The medical aspects of death, and the medical aspects of the human mind* (0,1278) supera a *Fragments of Ancient Poetry* (0,1230), *The Glebe 1914/09: Poems* (0,0870), *Home Poems* (0,0837) y *Notes and Queries* (0,0831). El primer documento es un tratado médico sobre la muerte y el alma, no poesía; el sistema lo recupera porque contiene los tres términos. Los puestos 2 a 4 sí son poesía. La consulta mezcla un término de género (`poems`) con dos temas (`soul`, `death`), y TF-IDF no modela esa distinción.

Los fragmentos se extraen alrededor del término de mayor IDF de cada consulta (con desempate por frecuencia total), de modo que muestran el contexto del término más discriminativo que aparece en cada documento.

**Verificación explícita vs. biblioteca.** La máxima diferencia absoluta entre las puntuaciones manuales y `cosine_similarity` es del orden de `1e-15` (el máximo observado es `2,99 × 10⁻¹⁵`), muy por debajo de la tolerancia de `1e-10`. Por tanto, ambas implementaciones producen las mismas puntuaciones y el mismo orden dentro del error de punto flotante.

### 7. Implementar BM25 y comparar rankings

BM25 es una función de ranking probabilística que mejora TF-IDF en dos aspectos: **satura el TF** (las repeticiones adicionales aportan cada vez menos) y **normaliza por la longitud del documento** de forma controlada por `b`. Su fórmula, para una consulta $q$ y un documento $d$, es:

$
\operatorname{BM25}(q,d)=\sum_{t\in q} IDF_{BM25}(t)\cdot
\frac{f(t,d)\,(k_1+1)}{f(t,d)+k_1\left(1-b+b\,\frac{|d|}{\operatorname{avgdl}}\right)}
$

con

$
IDF_{BM25}(t)=\ln\left(1+\frac{N-DF(t)+0,5}{DF(t)+0,5}\right)
$

Los componentes son:

- $f(t,d)$: frecuencia del término en el documento (TF).
- $|d|$: longitud del documento; $\operatorname{avgdl}$: longitud media del corpus.
- $k_1$: controla la saturación del TF (valores típicos entre 1,2 y 2,0).
- $b$: controla cuánto penaliza la longitud (0 = sin penalización, 1 = penalización completa).
- $IDF_{BM25}$: mide la rareza del término; es mayor para términos poco frecuentes.

Usaremos $k_1=1,5$ y $b=0,75$, los valores habituales por defecto. Para cada consulta se calcula la puntuación BM25 de los 1.000 documentos y se compara el top-5 con el de TF-IDF.

In [7]:
tf_rows, tf_cols, tf_values = [], [], []
for row, doc_id in enumerate(doc_ids):
    for term, count in tf_counts[doc_id].items():
        tf_rows.append(row)
        tf_cols.append(term_index[term])
        tf_values.append(count)

tf_matrix = sparse.csr_matrix(
    (tf_values, (tf_rows, tf_cols)),
    shape=(len(doc_ids), len(vocabulary)),
)

avgdl = float(doc_lengths.mean())
idf_bm25 = {
    term: math.log(1 + (len(doc_ids) - freq + 0.5) / (freq + 0.5))
    for term, freq in df.items()
}


def bm25_scores(query_tokens, k1=1.5, b=0.75):
    scores = np.zeros(len(doc_ids), dtype=float)
    for term in query_tokens:
        index = term_index.get(term)
        if index is None:
            continue
        frequency = tf_matrix[:, index].toarray().ravel().astype(float)
        denominator = frequency + k1 * (1 - b + b * doc_lengths / avgdl)
        scores += idf_bm25[term] * (frequency * (k1 + 1)) / np.where(
            denominator == 0, 1.0, denominator
        )
    return scores


bm25_rows = []
for label, query in queries.items():
    tokens = tokenize(query)
    scores = bm25_scores(tokens)
    tfidf_order = set(np.argsort(-query_data[label]['manual_scores'])[:5])
    bm25_order = np.argsort(-scores)[:5]
    overlap = len(tfidf_order & set(bm25_order))
    ranking_rows = []
    for position, doc_position in enumerate(bm25_order, start=1):
        ranking_rows.append(
            {
                'Posición': position,
                'Identificador': doc_ids[doc_position],
                'Título': titles[doc_ids[doc_position]],
                'BM25': round(float(scores[doc_position]), 6),
            }
        )
    print(f'Consulta: {query!r} — {label}')
    display(pd.DataFrame(ranking_rows))
    bm25_rows.append(
        {
            'Consulta': label,
            'Texto': query,
            'Coincidencias con TF-IDF en el top-5': f'{overlap}/5',
            'Mejor documento BM25': titles[doc_ids[int(np.argmax(scores))]],
            'Mejor BM25': round(float(scores.max()), 6),
        }
    )

display(pd.DataFrame(bm25_rows))

Consulta: 'love marriage family' — amor y matrimonio


,Posición,Identificador,Título,BM25
0,1,76066,Wedded by fate,4.075575
1,2,35723,Artist and Model (The Divorced Princess),4.075488
2,3,3622,The Duke's Children,4.050825
3,4,15774,"Ishmael; Or, In the Depths",4.033789
4,5,67406,The Husband's Story: A Novel,4.030775


Consulta: 'whale sea captain' — ballena y mar


,Posición,Identificador,Título,BM25
0,1,68958,Four years aboard the whaleship,8.697510
1,2,35518,The Ice Pilot,8.561258
2,3,32052,Grenfell: Knight-Errant of the North,8.385290
3,4,11573,"The Crater; Or, Vulcan's Peak: A Tale of the Pacific",8.307395
4,5,53560,With Rogers on the Frontier: A Story of 1756,7.487352


Consulta: 'king queen crown' — poder y corona


,Posición,Identificador,Título,BM25
0,1,163,Flower Fables,4.952517
1,2,74244,"The pageant of Parliament, vol. 1 of 2",4.936087
2,3,32675,The Strife of the Roses and Days of the Tudors in the West,4.932808
3,4,43428,"Kensington Palace, the birthplace of the Queen",4.920592
4,5,63088,The castles and abbeys of England; Vol. 1 of 2,4.913721


Consulta: 'poems soul death' — poesía y muerte


,Posición,Identificador,Título,BM25
0,1,8161,Fragments of Ancient Poetry,6.341434
1,2,13298,The California Birthday Book,6.243063
2,3,3545,The Complete Poetical Works of Oliver Goldsmith,6.237073
3,4,7007,The True Story of My Life: A Sketch,6.205540
4,5,34196,Porzia,6.203644


,Consulta,Texto,Coincidencias con TF-IDF en el top-5,Mejor documento BM25,Mejor BM25
0,amor y matrimonio,love marriage family,0/5,Wedded by fate,4.075575
1,ballena y mar,whale sea captain,1/5,Four years aboard the whaleship,8.697510
2,poder y corona,king queen crown,1/5,Flower Fables,4.952517
3,poesía y muerte,poems soul death,1/5,Fragments of Ancient Poetry,6.341434


**Interpretación de BM25 y comparación con TF-IDF.** BM25 y TF-IDF coinciden poco en el top-5: 0 de 5 en *amor y matrimonio* y 1 de 5 en las otras tres consultas. La diferencia se explica por los dos componentes propios de BM25:

- **Saturación del TF.** En TF-IDF el peso crece de forma lineal con el número de apariciones; en BM25 la fracción $f/(f+k_1(\cdot))$ se acerca a un máximo. Por eso un documento que repite mucho un término deja de acumular ventaja indefinidamente.
- **Normalización por longitud.** El factor $1-b+b\,|d|/\operatorname{avgdl}$ penaliza los documentos largos. En TF-IDF solo la similitud coseno normaliza longitudes; en BM25 la penalización se aplica directamente al término y es configurable con `b`.

El efecto es claro: para *amor y matrimonio*, BM25 sube *Wedded by fate*, *The Duke's Children* y *The Husband's Story: A Novel*, novelas que TF-IDF no colocaba en el top-5; para *ballena y mar* mantiene *Four years aboard the whaleship* en primer lugar y añade *The Ice Pilot* y *The Crater*. Para *poder y corona* BM25 prioriza *Flower Fables*, *The pageant of Parliament* y *The Strife of the Roses and Days of the Tudors in the West*. Para *poesía y muerte* coloca primero *Fragments of Ancient Poetry* y luego *The California Birthday Book* y *The Complete Poetical Works of Oliver Goldsmith*, más coherentes con la intención poética que el tratado médico que encabezaba TF-IDF.

En ambos modelos el resultado sigue siendo puramente léxico: no hay sinónimos, stemming, lematización ni comprensión del género discursivo. La comparación muestra que la elección de la función de ranking cambia el orden de los documentos, aunque todos los términos de las consultas existan en el vocabulario.

**Conclusiones**

1. La implementación explícita de TF-IDF coincide exactamente con `TfidfVectorizer` para TF crudo, IDF `ln(N/DF)+1`, sin normalización y sin TF sublineal, incluso con un vocabulario de 857.038 términos.
2. La similitud coseno hace comparables documentos de distinta longitud; sin esa normalización, los textos más extensos dominarían por acumulación de ocurrencias.
3. El corpus epub es mucho más grande y heterogéneo que la colección de 18 libros de NLTK: 1.000 documentos, 29,98 millones de tokens y libros en varios idiomas. Esto hace imprescindible usar matrices dispersas y un muestreo reproducible.
4. El modelo es puramente léxico: no reconoce sinónimos, morfología (`family` vs. `families`), polisemia ni intención. Por eso una consulta sobre amor y matrimonio puede devolver poesía, o una sobre realeza puede devolver un texto filosófico.
5. BM25 cambia el ranking al saturar el TF y penalizar la longitud de los documentos; en este corpus mejora la coherencia temática de varias consultas, pero sigue sin comprender el significado.
6. El ranking es interpretable y reproducible: las puntuaciones manuales y las de scikit-learn coinciden dentro de `1e-10`, y los fragmentos permiten verificar por qué se recuperó cada documento.